In [58]:
import sys
from pathlib import Path

import pandas as pd

data_path = Path('../Data/processed/census_cleaned.csv')

if not data_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df_cleaned = pd.read_csv(data_path)
df = df_cleaned.copy(deep=True)

In [59]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["income"])
# .str.strip()
# Removes leading and trailing whitespace from each string.
y = (df["income"].str.strip() == ">50K").astype(int)


# Split before fitting preprocessing transformations
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (36177, 13)
Testing shape: (9045, 13)


In [60]:
df.head()

,age,workclass,education_level,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K
1,50,Self-emp-not-inc,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,<=50K
2,38,Private,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K
3,53,Private,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K
4,28,Private,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K


In [61]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify numerical and categorical features
numerical_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

# Create preprocessing pipeline
# ColumnTransformer allows us to apply different transformations to different columns and combine their results into one outpu
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Fit on training data and transform
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [62]:
print(X_train.shape)
print(X_test.shape)

(36177, 103)
(9045, 103)


### Start Machine Learning Phase

### Start with Bagging Algorithms

In [63]:
# RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

random_forest_y_pred = random_forest_model.predict(X_test)


In [64]:
y_pred_rf = random_forest_model.predict(X_test)

In [65]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

random_forest_accuracy = accuracy_score(y_test, random_forest_y_pred)

random_forest_precision = precision_score(
    y_test, random_forest_y_pred, zero_division=0
)

random_forest_recall = recall_score(
    y_test, random_forest_y_pred, zero_division=0
)

random_forest_f1_score = f1_score(
    y_test, random_forest_y_pred, zero_division=0
)

random_forest_results = {
    "Algorithm": "Random Forest",
    "Accuracy": random_forest_accuracy,
    "Precision": random_forest_precision,
    "Recall": random_forest_recall,
    "F1_Score": random_forest_f1_score
}

random_forest_results

{'Algorithm': 'Random Forest',
 'Accuracy': 0.841901603095633,
 'Precision': 0.7077789150460594,
 'Recall': 0.6168599464763604,
 'F1_Score': 0.6591992373689228}

### Boosting Algorithms

In [66]:
# ADABOOST
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2),
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

adaboost_model.fit(X_train, y_train)

adaboost_y_pred = adaboost_model.predict(X_test)

In [67]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

adaboost_accuracy = accuracy_score(y_test, adaboost_y_pred)

adaboost_precision = precision_score(
    y_test, adaboost_y_pred, zero_division=0
)

adaboost_recall = recall_score(
    y_test, adaboost_y_pred, zero_division=0
)

adaboost_f1_score = f1_score(
    y_test, adaboost_y_pred, zero_division=0
)

adaboost_results = {
    "Algorithm": "AdaBoost",
    "Accuracy": adaboost_accuracy,
    "Precision": adaboost_precision,
    "Recall": adaboost_recall,
    "F1_Score": adaboost_f1_score
}

adaboost_results

{'Algorithm': 'AdaBoost',
 'Accuracy': 0.849640685461581,
 'Precision': 0.7647058823529411,
 'Recall': 0.568242640499554,
 'F1_Score': 0.6519959058341863}

In [68]:
# GRADIENT BOOSTING MODEL
from sklearn.ensemble import GradientBoostingClassifier

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0,
    random_state=42
)

gradient_boosting_model.fit(X_train, y_train)

gradient_boosting_y_pred = (
    gradient_boosting_model.predict(X_test)
)

In [69]:
gradient_boosting_accuracy = accuracy_score(
    y_test, gradient_boosting_y_pred
)

gradient_boosting_precision = precision_score(
    y_test, gradient_boosting_y_pred, zero_division=0
)

gradient_boosting_recall = recall_score(
    y_test, gradient_boosting_y_pred, zero_division=0
)

gradient_boosting_f1_score = f1_score(
    y_test, gradient_boosting_y_pred, zero_division=0
)

gradient_boosting_results = {
    "Algorithm": "Gradient Boosting",
    "Accuracy": gradient_boosting_accuracy,
    "Precision": gradient_boosting_precision,
    "Recall": gradient_boosting_recall,
    "F1_Score": gradient_boosting_f1_score
}

gradient_boosting_results

{'Algorithm': 'Gradient Boosting',
 'Accuracy': 0.8603648424543947,
 'Precision': 0.7881106533254856,
 'Recall': 0.5972346119536128,
 'F1_Score': 0.6795229637147931}

In [70]:
# XGBoost
from xgboost import XGBClassifier

xgboost_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    # Uses all training rows per boosting iteration
    subsample=1.0,
    # Uses all available features when constructing each tree
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgboost_model.fit(X_train, y_train)

xgboost_y_pred = xgboost_model.predict(X_test)

In [71]:
xgboost_accuracy = accuracy_score(
    y_test, xgboost_y_pred
)

xgboost_precision = precision_score(
    y_test, xgboost_y_pred, zero_division=0
)

xgboost_recall = recall_score(
    y_test, xgboost_y_pred, zero_division=0
)

xgboost_f1_score = f1_score(
    y_test, xgboost_y_pred, zero_division=0
)

xgboost_results = {
    "Algorithm": "XGBoost",
    "Accuracy": xgboost_accuracy,
    "Precision": xgboost_precision,
    "Recall": xgboost_recall,
    "F1_Score": xgboost_f1_score
}

xgboost_results

{'Algorithm': 'XGBoost',
 'Accuracy': 0.8569375345494749,
 'Precision': 0.7869249394673123,
 'Recall': 0.5798394290811775,
 'F1_Score': 0.6676938880328711}

In [72]:
# COMPARE FOUR ESEMBLE LEARNING ALGORITHMS
evaluation_results_df = pd.DataFrame([
    random_forest_results,
    adaboost_results,
    gradient_boosting_results,
    xgboost_results
])

evaluation_results_df = evaluation_results_df.sort_values(
    by="F1_Score",
    ascending=False
).reset_index(drop=True)

evaluation_results_df

,Algorithm,Accuracy,Precision,Recall,F1_Score
0,Gradient Boosting,0.860365,0.788111,0.597235,0.679523
1,XGBoost,0.856938,0.786925,0.579839,0.667694
2,Random Forest,0.841902,0.707779,0.616860,0.659199
3,AdaBoost,0.849641,0.764706,0.568243,0.651996


In [73]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

models = {
    "Random Forest": random_forest_model,
    "AdaBoost": adaboost_model,
    "Gradient Boosting": gradient_boosting_model,
    "XGBoost": xgboost_model
}

evaluation_details = []

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    evaluation_details.append({
        "Algorithm": model_name,
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    })

    print(f"\n{'=' * 50}")
    print(f"Model: {model_name}")
    print("=" * 50)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=["<=50K", ">50K"]
    ))

roc_auc_results_df = pd.DataFrame(evaluation_details)

roc_auc_results_df = roc_auc_results_df.sort_values(
    by="ROC_AUC",
    ascending=False
).reset_index(drop=True)

roc_auc_results_df


Model: Random Forest

Confusion Matrix:
[[6232  571]
 [ 859 1383]]

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.88      0.92      0.90      6803
        >50K       0.71      0.62      0.66      2242

    accuracy                           0.84      9045
   macro avg       0.79      0.77      0.78      9045
weighted avg       0.84      0.84      0.84      9045


Model: AdaBoost

Confusion Matrix:
[[6411  392]
 [ 968 1274]]

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.87      0.94      0.90      6803
        >50K       0.76      0.57      0.65      2242

    accuracy                           0.85      9045
   macro avg       0.82      0.76      0.78      9045
weighted avg       0.84      0.85      0.84      9045


Model: Gradient Boosting

Confusion Matrix:
[[6443  360]
 [ 903 1339]]

Classification Report:
              precision    recall  f1-score   support

       <=50K    

,Algorithm,ROC_AUC
0,Gradient Boosting,0.916771
1,XGBoost,0.915611
2,AdaBoost,0.903497
3,Random Forest,0.889959


In [76]:
import joblib

joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(xgboost_model, "model.pkl")

['model.pkl']